# Campus Waste Segregation Assistant - training on Colab / Kaggle

Use this only if you have no local GPU (PLAN.md Section 12). Runtime -> Change runtime type -> **GPU**.
At the end, download `models/<run>/best.pt` and the `reports/` folder into your local repo.

In [ ]:
# 1. Get the code (replace with your repository URL, or upload a zip of the repo)
!git clone https://github.com/<your-user>/waste-segregation-assistant.git
%cd waste-segregation-assistant
!pip install -q imagehash grad-cam kaggle onnx onnxruntime
!pip install -q -e .

In [ ]:
# 2. Kaggle credentials: upload kaggle.json when prompted
from google.colab import files
import os, shutil
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
# 3. Data: download, resize, de-duplicate, split, EDA
!python scripts/download_data.py --trashnet
!python -m waste.data.prepare --config configs/data.yaml --workers 2

In [ ]:
# 4. Train baseline and MobileNetV2 (plus optional ablations)
!python -m waste.train --config configs/train_custom_cnn.yaml --num-workers 2
!python -m waste.train --config configs/train_mobilenet.yaml --num-workers 2
# !python -m waste.train --config configs/train_mobilenet_noaug.yaml --num-workers 2
# !python -m waste.train --config configs/train_mobilenet_headonly.yaml --num-workers 2
# !python -m waste.train --config configs/train_resnet18.yaml --num-workers 2

In [ ]:
# 5. Evaluate, tune the threshold, compare
!python -m waste.evaluate all --split test
!python -m waste.evaluate coverage --run mobilenet_v2 --write-config
!python -m waste.evaluate external --run mobilenet_v2 --dataset trashnet
!python -m waste.evaluate compare

In [ ]:
# 6. Package the results for download
!zip -qr results.zip models reports configs/inference.yaml
files.download('results.zip')